# LSTM configurable: comparación de features

Ejecuta el pipeline de LSTM variando el conjunto de features y reporta métricas en escala original (MAE, MSE, R²) ponderadas por muestras, además de globales y por ticker.


In [1]:
# Imports y configuración
%pip install --quiet numpy pandas scikit-learn matplotlib torch

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from IPython.display import display

from lstm_pipeline import run_experiment

DATA_DIR = "/Users/mariasvidal/Desktop/LSTM-for-Time-Series-Prediction/clean_new_model/dataset_ceci_sol"

SEQ = 5
SPLIT = 0.75
BATCH = 16
EPOCHS = 20
LR = 1e-3
HIDDEN = 32
LAYERS = 1
DROPOUT = 0.0

Note: you may need to restart the kernel to use updated packages.


# Experimento 1: Comparo entrenar con y sin sentiment analysis creado por nosotras

In [2]:
TICKERS_TRAIN = ['AAPL', 'AMD', 'AMZN', 'DIS', 'F', 'GOOG', 'GS', 'KO', 'MSFT', 'NVDA', 'SPY', 'TSLA', 'UBER', 'WMT', 'BRK-B']
TICKERS_TEST = ['AAPL', 'AMD', 'AMZN', 'DIS', 'F', 'GOOG', 'GS', 'KO', 'MSFT', 'NVDA', 'SPY', 'TSLA', 'UBER', 'WMT', 'BRK-B']

In [3]:
FEATURE_SETS = [
     ["Adj Close", "High","Low","Volume","VIX"],
     ["puntaje_sent_fin_pos","puntaje_sent_fin_neu","puntaje_sent_fin_neg", "Adj Close", "High","Low","Volume","VIX","has_news"],
     ["puntaje_sent_fin_pos","puntaje_sent_fin_neg", "Adj Close", "High","Low","Volume","VIX","has_news"],
     ["recom_inv_llm", "Adj Close", "High","Low","Volume","VIX","has_news"],
     ["recom_inv_llm","puntaje_sent_fin_pos","puntaje_sent_fin_neu","puntaje_sent_fin_neg", "Adj Close", "High","Low","Volume","VIX","has_news"],
     ["Sentiment_gpt","News_flag", "Adj Close", "High","Low","Volume","VIX"]
]

In [4]:
FEATURE_TO_SCALE = [
    ["Adj Close", "High","Low","Volume"],
    ["Adj Close", "High","Low","Volume"],
    ["Adj Close", "High","Low","Volume"],
    ["Adj Close", "High","Low","Volume"],
    ["Adj Close", "High","Low","Volume"],
    ["Adj Close", "High","Low","Volume"],
]

In [ ]:
SEEDS_TK = SEEDS = [10, 12, 20, 33, 42, 56, 60, 70, 88, 32, 90, 100, 96, 66]
metrics = [ "MSE_w","NRMSE_w", "WMAPE_w"]
rows_tk = []
rows = []
for feats_idx in range(len(FEATURE_SETS)):
    feats = FEATURE_SETS[feats_idx]
    feats_scale = FEATURE_TO_SCALE[feats_idx] if 'FEATURE_TO_SCALE' in globals() else feats
    for sd in SEEDS_TK:
        if feats == ["Sentiment_gpt","News_flag", "Adj Close", "High","Low","Volume","VIX"]:
           # no estan todos los tickers con el sentiment de paper
            TICKERS_TRAIN = ['AAPL', 'AMD', 'AMZN', 'DIS',  'GOOG',  'KO', 'MSFT', 'NVDA',  'TSLA',  'WMT', 'BRK-B']
            TICKERS_TEST =['AAPL', 'AMD', 'AMZN', 'DIS',  'GOOG',  'KO', 'MSFT', 'NVDA',  'TSLA',  'WMT', 'BRK-B']
        else:
            TICKERS_TRAIN = ['AAPL', 'AMD', 'AMZN', 'DIS', 'F', 'GOOG', 'GS', 'KO', 'MSFT', 'NVDA', 'SPY', 'TSLA', 'UBER', 'WMT', 'BRK-B']
            TICKERS_TEST = ['AAPL', 'AMD', 'AMZN', 'DIS', 'F', 'GOOG', 'GS', 'KO', 'MSFT', 'NVDA', 'SPY', 'TSLA', 'UBER', 'WMT', 'BRK-B']
        out = run_experiment(
            data_dir=DATA_DIR,
            features=feats,
            features_to_scale=feats_scale,
            tickers=TICKERS_TRAIN,
            test_tickers=TICKERS_TEST,
            seq=SEQ,
            split=SPLIT,
            batch_size=BATCH,
            epochs=EPOCHS,
            hidden_dim=HIDDEN,
            num_layers=LAYERS,
            dropout=DROPOUT,
            lr=LR,
            device=("cuda" if torch.cuda.is_available() else "cpu"),
            seed=sd,
            early_stopping=True,
            patience=5,
        )
        rows.append({
            'features': ', '.join(feats),
            'seed': sd,
            'MAE_w': out['weighted_metrics'].get('MAE_weighted'),
            'MSE_w': out['weighted_metrics'].get('MSE_weighted'),
            'NRMSE_w': out['weighted_metrics'].get('NRMSE_weighted'),
            'R2_w': out['weighted_metrics'].get('R2_weighted'),
            'WMAPE_w': out['weighted_metrics'].get('WMAPE_weighted'),
            'HIT_RATE_w': out['weighted_metrics'].get('HIT_RATE_weighted'),
        })
        for tik, met in out['per_ticker'].items():
            rows_tk.append({
                'features': ', '.join(feats),
                'seed': sd,
                'ticker': tik,
                'NRMSE': met.get('NRMSE'),
                'MSE': met.get('MSE'),
                'WMAPE': met.get('WMAPE'),
            })

seed_results = pd.DataFrame(rows) # tamaño nro de seeds * nro de set de features

# Agregación mean/std por features
agg_dict = {m: ['mean', 'std'] for m in metrics}
agg = seed_results.groupby('features').agg(agg_dict).reset_index()
# Aplanar columnas
agg.columns = [
    'features'
] + [f"{m}_{stat}" for m, stats in agg_dict.items() for stat in stats]



per_ticker_seed = pd.DataFrame(rows_tk)
agg_ticker = per_ticker_seed.groupby(['features','ticker']).agg({'MSE':['mean','std'],'NRMSE':['mean','std'], 'WMAPE':['mean','std']}).reset_index()
agg_ticker.columns = ['features','ticker','MSE_mean','MSE_std','NRMSE_mean','NRMSE_std','WMAPE_mean','WMAPE_std']

# DFs por ticker: creo un df por cada ticker con las metricas de cada set de features


In [10]:
dfs_por_ticker: dict[str, pd.DataFrame] = {}
for tk in sorted(agg_ticker['ticker'].unique()):
    df_tk = (
        agg_ticker.loc[agg_ticker['ticker'] == tk, ['features', 'MSE_mean', 'MSE_std', 'WMAPE_mean', 'WMAPE_std']]
        .sort_values([ 'WMAPE_mean'])
        .reset_index(drop=True)
    )
    dfs_por_ticker[tk] = df_tk

# Agregar columna relativa por ticker: (MSE_mean - MSE_baseline) / MSE_baseline

In [11]:
# Baseline: features == "Adj Close, High, Low, Volume, VIX"

baseline_fs = "Adj Close, High, Low, Volume, VIX"
for tk, df in dfs_por_ticker.items():
    try:
        base_row = df.loc[df['features'] == baseline_fs]
        if not base_row.empty and pd.notna(base_row['MSE_mean'].values[0]):
            base = float(base_row['MSE_mean'].values[0])
            if base != 0:
                df['Mejora MSE respecto a baseline (%)'] = round(100*(base -df['MSE_mean'] ) / base,2)
            else:
                df['Mejora MSE respecto a baseline (%)'] = np.nan
        else:
            df['Mejora MSE respecto a baseline'] = np.nan
        dfs_por_ticker[tk] = df
    except Exception as e:
        print(f"No se pudo calcular Mejora MSE respecto a baseline para {tk}: {e}")
print("Columna 'MSE_rel_vs_baseline' agregada a cada dfs_por_ticker[tk].")


Columna 'MSE_rel_vs_baseline' agregada a cada dfs_por_ticker[tk].


# Extraer features con mayor mejora de MSE (%) respecto al baseline por ticker


In [12]:
# baseline: "Adj Close, High, Low, ():Volume, VIX"

baseline_fs = "Adj Close, High, Low, Volume, VIX"
best_features_by_ticker: dict[str, pd.DataFrame] = {}
for tk, df in dfs_por_ticker.items():
    # Asegurar cómputo de mejora (%) aunque no exista la columna previa
    base_row = df.loc[df['features'] == baseline_fs]
    if base_row.empty or pd.isna(base_row['MSE_mean'].values[0]):
        # Sin baseline, no se puede calcular mejora
        best_features_by_ticker[tk] = df.iloc[0:0].copy()
        continue
    base = float(base_row['MSE_mean'].values[0])
    if base == 0:
        best_features_by_ticker[tk] = df.iloc[0:0].copy()
        continue
    df = df.copy()
    df['MSE_improvement_pct'] = (base - df['MSE_mean']) / base * 100.0
    # Excluir baseline de la selección y filtrar mejoras positivas
    df_no_base = df.loc[df['features'] != baseline_fs]
    if df_no_base.empty:
        best_features_by_ticker[tk] = df.iloc[0:0].copy()
        continue
    max_imp = df_no_base['MSE_improvement_pct'].max()
    if pd.isna(max_imp) or max_imp <= 0:
        # Si ninguna configuración mejora el baseline, devolver vacío
        best_features_by_ticker[tk] = df_no_base.sort_values('MSE_improvement_pct', ascending=False).head(0)
    else:
        best_features_by_ticker[tk] = df_no_base.loc[df_no_base['MSE_improvement_pct'] == max_imp].sort_values('MSE_improvement_pct', ascending=False)
print("Diccionario creado: best_features_by_ticker[ticker]")
# Ejemplo de acceso: best_features_by_ticker['AAPL']


Diccionario creado: best_features_by_ticker[ticker]


#  Cuántas veces aparece cada set de features como el set de features con mejor prediccion para un ticker

In [13]:

from collections import Counter
global_counter = Counter()
freq_by_ticker: dict[str, pd.Series] = {}


for tk, df in best_features_by_ticker.items():
    if df is None or len(df) == 0 or 'features' not in df.columns:
        continue
    counts = df['features'].value_counts()
   
    freq_by_ticker[tk] = counts
    for feat, cnt in counts.items():
        global_counter[feat] += int(cnt)

best_features_counts = (
    pd.Series(global_counter)
    .sort_values(ascending=False)
    .rename('count')
    .to_frame()
    .reset_index()
    .rename(columns={'index': 'features'})
)

print("Conteo global de features en los mejores por ticker (desc):")
display(best_features_counts)


Conteo global de features en los mejores por ticker (desc):


,features,count
0,"recom_inv_llm, Adj Close, High, Low, Volume, V...",8
1,"Sentiment_gpt, News_flag, Adj Close, High, Low...",3
2,"puntaje_sent_fin_pos, puntaje_sent_fin_neu, pu...",2
3,"puntaje_sent_fin_pos, puntaje_sent_fin_neg, Ad...",1


# Solo ejecutar para guardar los csvs de resultados

In [60]:
# Guardar cada DataFrame de dfs_por_ticker en CSV con el nombre del ticker
"""
from pathlib import Path
out_dir = Path('dfs_por_ticker_csv')
out_dir.mkdir(exist_ok=True, parents=True)
saved = 0
for tk, df in dfs_por_ticker.items():
    if df is None or len(df) == 0:
        continue
    safe_name = str(tk).replace('/', '_').replace('\\', '_')
    df.to_csv(out_dir / f"{safe_name}.csv", index=False)
    saved += 1
print(f"Guardados {saved} CSVs en: {out_dir.resolve()}")
"""

Guardados 15 CSVs en: /Users/mariasvidal/Desktop/LSTM-for-Time-Series-Prediction/clean_new_model/dfs_por_ticker_csv


# Metricas globales concatenando todos los tickers (pesadas por la cantidad de puntos en cada ticker)

In [8]:
agg.sort_values(('MSE_w_mean'), ascending=True).reset_index(drop=True)

,features,MSE_w_mean,MSE_w_std,NRMSE_w_mean,NRMSE_w_std,WMAPE_w_mean,WMAPE_w_std
0,"recom_inv_llm, Adj Close, High, Low, Volume, V...",11.645673,2.458009,0.699746,0.037594,0.012362,0.001189
1,"puntaje_sent_fin_pos, puntaje_sent_fin_neg, Ad...",23.768184,7.522009,0.769965,0.092894,0.016178,0.002489
2,"Sentiment_gpt, News_flag, Adj Close, High, Low...",31.391350,7.737729,0.776898,0.068107,0.018756,0.002056
3,"Adj Close, High, Low, Volume, VIX",36.456127,8.408592,0.976982,0.092162,0.021079,0.002381
4,"puntaje_sent_fin_pos, puntaje_sent_fin_neu, pu...",55.692846,10.583446,1.287606,0.111872,0.025135,0.002172
5,"recom_inv_llm, puntaje_sent_fin_pos, puntaje_s...",2940.863461,49.618892,1.183389,0.023605,0.170077,0.003237
